# 04. Define a lepton

Explore how lepton selections change signal and background acceptance using
full-pipeline ROOT samples with retained Electron or Muon collections.

These cuts select reconstructed Delphes candidates; they do not rerun detector
reconstruction or recover candidates already discarded. The results describe
this simulation, not measured experimental fake rates.

Copy this notebook into `work`, select **Python (hep)**, and run cells from the
top. Choose the runs specified in your assignment.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from findingz.catalog import load_catalog
from findingz.delphes import default_run_root
from findingz.hypotheses import build_sample_library

library = build_sample_library(load_catalog(), default_run_root())
display(pd.DataFrame([{"sample_id": k, "label": s.label, "cross_section_pb": s.cross_section_pb,
                      "events": s.generated_events, "config": s.config} for k, s in library.items()]))
def get_sample(sample_id):
    if sample_id not in library:
        raise ValueError(f"Sample {sample_id!r} is unavailable. Choose an ID from the table above.")
    return library[sample_id]
def weights(frame):
    return pd.to_numeric(frame.get("weight", pd.Series(1., index=frame.index)))


In [ ]:
from findingz.delphes import open_run
import awkward as ak
run_ids = []  # raw run IDs (without the library's "run:" prefix): signal and background
collection = "muons"  # or "electrons" with an electron sample
run_root = None  # for shared samples, set the parent directory of their run folders
pt_minima = [5.,10.,20.,30.]
eta_max = 2.5
isolation_max = None  # e.g. 0.15, only if the retained collection has isolation
for run_id in run_ids:
    events = open_run(run_id, run_root=run_root)
    objects = events.collection(collection)
    print(run_id, "available fields:", ak.fields(objects))
    rows = []
    for threshold in pt_minima:
        keep = (objects.pt > threshold) & (abs(objects.eta) < eta_max)
        if isolation_max is not None:
            if "isolation" not in ak.fields(objects):
                raise ValueError("Isolation is not retained in this sample.")
            keep = keep & (objects.isolation < isolation_max)
        selected = objects[keep]
        opposite_sign = (ak.any(selected.charge<0,axis=1) & ak.any(selected.charge>0,axis=1))
        rows.append({"pt_min":threshold, "opposite_sign_events":int(ak.sum(opposite_sign)),
                     "fraction_of_ROOT_events":float(ak.mean(opposite_sign))})
    display(pd.DataFrame(rows))

## Questions and submission
1. Plot signal efficiency against background acceptance for the same definitions.
2. Explain the normalization denominator and how it differs from an object efficiency.
3. Why can tightening isolation in this notebook not undo a cut already made by Delphes?
4. Extension: form opposite-sign pairs from selected objects and reconstruct mll; specify a rule if multiple pairs exist.

Submit your edited notebook with figures, units, sample IDs, generation settings, and a short interpretation. Do not equate Monte Carlo event count with experimental luminosity.